# 15 - Inference and submission

Runs with **internet OFF** (code-competition rules). Attach:
1. the competition data;
2. ONE model bundle (`model/config.json`): notebook 16's output (fine-tuned 2.5D MIL,
   `model_kind: "mil"`) or notebook 14's output (frozen ResNet-18 + linear heads);
3. a Kaggle dataset containing the Med_Im_Prepper source folder `medprep/`
   (upload the repository's `medprep` directory, or a wheel built with
   `python -m pip wheel . --no-deps -w dist` from the Med_Im_Prepper root).

Preprocessing and feature code are the same cells as notebooks 13 and 14; the
version strings saved by 14 are checked before anything runs. A valid fallback
`submission.csv` is written first so a crash never leaves no file.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# The competition mount path has differed between Kaggle runtimes; take the first that has train.csv.
_SLUG = "rsna-knee-abnormality-detection"
_roots = [Path("/kaggle/input/competitions") / _SLUG, Path("/kaggle/input") / _SLUG,
          *sorted(Path("/kaggle/input").glob("*/" + _SLUG)), *sorted(Path("/kaggle/input").glob("*"))]
ROOT = next((r for r in _roots if (r / "train.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("competition train.csv not found; /kaggle/input has: "
                            + str(sorted(str(p) for p in Path("/kaggle/input").glob("*/*"))[:50]))
print("competition data:", ROOT)
OUT = Path("/kaggle/working")
TARGET_COLUMNS = [
    "ACL", "MCL",
    "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]
SEED = 42
from pathlib import Path


def find_input(name):
    """Locate one attached upstream file by name.

    Only notebook outputs and datasets are searched: walking the competition
    mount would visit millions of DICOM files.
    """
    hits = []
    for base in (Path("/kaggle/input/notebooks"), Path("/kaggle/input/datasets"),
                 Path("/kaggle/input")):
        if base.is_dir():
            for child in base.iterdir():
                if (child.is_dir()
                        and child.name not in ("competitions", "notebooks", "datasets")
                        and not (child / "train_series").exists()
                        and not (child / "test_series").exists()):
                    hits += list(child.rglob(name))
    hits = sorted(set(hits))
    if len(hits) != 1:
        raise FileNotFoundError(
            f"expected exactly one attached '{name}', found {len(hits)}: {hits[:5]}. "
            "Attach the upstream notebook output via Add Input."
        )
    return hits[0]

import glob
import json
import subprocess
import sys

try:
    import medprep  # noqa: F401
except ImportError:
    wheels = glob.glob("/kaggle/input/**/medprep_starter-*.whl", recursive=True)
    if wheels:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-index",
                        "-q", wheels[0]], check=True)
    else:
        src = find_input("medprep").parent
        sys.path.insert(0, str(src))
import medprep
print("medprep", medprep.__version__)

test = pd.read_csv(ROOT / "test.csv", dtype={"StudyInstanceUID": str})
sample = pd.read_csv(ROOT / "sample_submission.csv", nrows=1)
assert list(sample.columns) == ["StudyInstanceUID"] + TARGET_COLUMNS

MODEL = find_input("config.json").parent
config = json.loads((MODEL / "config.json").read_text())
assert config["target_columns"] == TARGET_COLUMNS
fallback = pd.DataFrame({"StudyInstanceUID": test["StudyInstanceUID"],
                         **{t: config["fallback_prevalence"][t] for t in TARGET_COLUMNS}})
fallback.to_csv(OUT / "submission.csv", index=False)
print("fallback submission written:", len(fallback), "rows")

In [ ]:
# Shared preprocessing: this cell is identical in notebooks 13 and 15.
# Change PREPROC_VERSION whenever anything below changes; 15 refuses to run
# against a model trained on a different version.
import numpy as np
import pydicom
from medprep import preprocess, PreprocessConfig, load_image

PREPROC_VERSION = "v2-16x224-pad-p1_99-canonical-lps-leftknee"
PLANES = ("Sagittal", "Coronal", "Axial")
N_SLICES = 16
IMG = 224
LAT_MIN_OFFSET_MM = 20.0  # closer to the midline than this: do not guess the side


def _floats(value, n):
    out = np.asarray([float(x) for x in value], dtype=float)
    if out.shape != (n,):
        raise ValueError(f"expected {n} values, got {out.shape}")
    return out


def select_series(study_series, series_root):
    """One series per plane: fluid-sensitive + fat-suppressed preferred (as in 06),
    then most slices, then UID for a deterministic tie-break."""
    picks = {}
    for plane in PLANES:
        g = study_series[study_series["Anatomical_Plane"].eq(plane)]
        if g.empty:
            continue
        ranked = []
        for row in g.itertuples(index=False):
            folder = series_root / row.StudyInstanceUID / row.SeriesInstanceUID
            n = sum(1 for _ in folder.glob("*.dcm"))
            pref = int(row.Fluid_Sensitive == 1 and row.Fat_Suppression == 1)
            ranked.append((-pref, -n, row.SeriesInstanceUID, folder, n))
        best = sorted(ranked)[0]
        if best[4] >= 2:
            picks[plane] = best[3]
    return picks


def canonical_axes(iop):
    """Patient axis carried by (slice, row, column) array axes, and their signs.

    DICOM: iop[0:3] is the direction of increasing column index (array axis 2),
    iop[3:6] of increasing row index (array axis 1); slices are sorted below by
    ascending patient coordinate, so axis 0 is always positive.
    """
    row_dir, col_dir = iop[:3], iop[3:]
    normal = np.cross(row_dir, col_dir)
    axes = (int(np.argmax(np.abs(normal))), int(np.argmax(np.abs(col_dir))),
            int(np.argmax(np.abs(row_dir))))
    if len(set(axes)) != 3:
        raise ValueError(f"oblique orientation, axes collide: {axes}")
    signs = (1, int(np.sign(col_dir[axes[1]])), int(np.sign(row_dir[axes[2]])))
    return axes, signs


def to_canonical(vol, iop):
    """Flip (never rotate) so every array axis increases along +L, +P or +S."""
    axes, signs = canonical_axes(iop)
    if signs[1] < 0:
        vol = vol[:, ::-1, :]
    if signs[2] < 0:
        vol = vol[:, :, ::-1]
    return np.ascontiguousarray(vol), axes


def to_left_knee(vol, axes, side):
    """Mirror a right knee across the patient x axis (LPS +x = patient left)."""
    if side != "R":
        return vol
    return np.ascontiguousarray(np.flip(vol, axis=axes.index(0)))


def read_series(folder):
    """Ordered, LUT-applied, canonically flipped float32 volume plus geometry."""
    files = sorted(folder.glob("*.dcm"))
    headers = [pydicom.dcmread(f, stop_before_pixels=True) for f in files]
    for ds in headers:
        if str(getattr(ds, "PhotometricInterpretation", "")) != "MONOCHROME2":
            raise ValueError("non-MONOCHROME2 slice; medprep does not invert MONOCHROME1")
    iops = np.stack([_floats(ds.ImageOrientationPatient, 6) for ds in headers])
    if not np.allclose(iops, iops[0], atol=1e-3, rtol=0):
        raise ValueError("slice orientations differ within the series")
    iop = iops[0]
    axes, _ = canonical_axes(iop)
    ipps = np.stack([_floats(ds.ImagePositionPatient, 3) for ds in headers])
    order = np.argsort(ipps[:, axes[0]], kind="stable")
    # medprep.load_image applies the modality LUT and rejects padding/multiframe.
    slices = [load_image(files[i])[0] for i in order]
    if len({s.shape for s in slices}) != 1:
        raise ValueError("slice shapes differ within the series")
    vol, axes = to_canonical(np.stack(slices).astype(np.float32), iop)
    mid = headers[order[len(order) // 2]]
    ps = _floats(mid.PixelSpacing, 2)
    centre = (_floats(mid.ImagePositionPatient, 3)
              + iop[:3] * ps[1] * float(mid.Columns) / 2
              + iop[3:] * ps[0] * float(mid.Rows) / 2)
    tag = str(getattr(mid, "Laterality", "") or "").strip().upper()
    tag = {"LEFT": "L", "RIGHT": "R"}.get(tag, tag)  # some sites write the full word
    return vol, {"axes": axes, "centre_x": float(centre[0]), "tag": tag,
                 "n_slices": len(files),
                 "geom_plane": {0: "Sagittal", 1: "Coronal", 2: "Axial"}[axes[0]]}


def study_side(metas):
    """Laterality tag if consistent, else median image-centre x; None if unsure."""
    tags = {m["tag"] for m in metas} - {""}
    if len(tags) == 1 and tags <= {"L", "R"}:
        return tags.pop(), "tag"
    xs = [m["centre_x"] for m in metas]
    if not xs:
        return None, "none"
    x = float(np.median(xs))
    if abs(x) < LAT_MIN_OFFSET_MM:
        return None, "unresolved"
    return ("R" if x < 0 else "L"), "geometry"


NORMALISE = PreprocessConfig(spatial_dims=3, percentiles=(1, 99), normalization="minmax")
RESIZE = PreprocessConfig(spatial_dims=3, target_shape=(N_SLICES, IMG, IMG), resize_mode="pad")


def finish_volume(vol):
    """Per-volume 1-99% clip + minmax (nothing fitted on data), 16 evenly spaced
    slices, aspect-preserving resize/pad to 224, uint8."""
    vol = preprocess(vol, config=NORMALISE)["image"]
    idx = np.round(np.linspace(0, len(vol) - 1, N_SLICES)).astype(int)
    vol = preprocess(np.ascontiguousarray(vol[idx]), config=RESIZE)["image"]
    return np.round(np.clip(vol, 0, 1) * 255).astype(np.uint8)


def process_study(study_series, series_root):
    """(3, 16, 224, 224) uint8 in PLANES order, presence mask, log record."""
    picks = select_series(study_series, series_root)
    out = np.zeros((len(PLANES), N_SLICES, IMG, IMG), np.uint8)
    present = np.zeros(len(PLANES), np.float32)
    raw, metas, errors = {}, [], []
    for plane, folder in picks.items():
        try:
            vol, meta = read_series(folder)
            raw[plane] = (vol, meta)
            metas.append(meta)
        except Exception as exc:
            errors.append(f"{plane}: {type(exc).__name__}: {exc}")
    side, side_source = study_side(metas)
    plane_mismatch = 0
    for k, plane in enumerate(PLANES):
        if plane not in raw:
            continue
        vol, meta = raw[plane]
        plane_mismatch += int(meta["geom_plane"] != plane)
        out[k] = finish_volume(to_left_knee(vol, meta["axes"], side))
        present[k] = 1.0
    record = {"side": side or "", "side_source": side_source,
              "planes_present": int(present.sum()), "plane_mismatch": plane_mismatch,
              "error": " | ".join(errors)}
    return out, present, record

In [ ]:
# Shared feature extractor: identical in notebooks 14 and 15.
import torch
from torch import nn
from torchvision.models import resnet18

FEATURE_VERSION = "resnet18-imagenet-frozen-slice-mean-3planes+presence"
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def build_encoder(state_dict=None):
    from torchvision.models import ResNet18_Weights
    weights = None if state_dict is not None else ResNet18_Weights.IMAGENET1K_V1
    enc = resnet18(weights=weights)
    enc.fc = nn.Identity()
    if state_dict is not None:
        enc.load_state_dict(state_dict)
    return enc.eval().requires_grad_(False)


@torch.inference_mode()
def study_features(encoder, volumes, present, device):
    """volumes: (B, 3, 16, 224, 224) uint8; present: (B, 3).
    Returns (B, 3*512 + 3): per-plane slice-mean features, zeros for a missing
    plane, followed by the presence mask."""
    b, p, d, h, w = volumes.shape
    x = volumes.to(device).reshape(b * p * d, 1, h, w).float().div_(255)
    x = (x.repeat(1, 3, 1, 1) - IMAGENET_MEAN.to(device)) / IMAGENET_STD.to(device)
    with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
        f = encoder(x)
    f = f.float().reshape(b, p, d, -1).mean(dim=2) * present.to(device)[..., None]
    return torch.cat([f.reshape(b, -1), present.to(device)], dim=1).cpu()

In [ ]:
# Shared 2.5D attention-MIL model: identical in notebooks 16 and 15.
import numpy as np
import timm
import torch
from torch import nn

MIL_VERSION = "mil-v1-2p5d-3slice-perfinding-attn-planeemb"
N_PLANES, CACHE_SLICES = 3, 16
WINDOW_CENTRES = np.arange(1, CACHE_SLICES - 1)  # a window is slices c-1, c, c+1


def eval_centres(k):
    """k window centres spread evenly over the volume (deterministic)."""
    return WINDOW_CENTRES[np.round(np.linspace(0, len(WINDOW_CENTRES) - 1, k)).astype(int)]


def windows_from_volume(volume, centres_per_plane):
    """volume: (3, 16, H, W) uint8 -> (3, K, 3, H, W) float32 in [0, 1]."""
    out = np.stack([np.stack([volume[p, c - 1:c + 2] for c in centres_per_plane[p]])
                    for p in range(N_PLANES)])
    return out.astype(np.float32) / 255.0


class KneeMIL(nn.Module):
    """Backbone per 3-slice window + plane embedding, then one attention pooling
    per finding over all windows of the study (missing planes masked out)."""

    def __init__(self, arch, n_out=12, pretrained=True, drop=0.2):
        super().__init__()
        self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0)
        f = self.backbone.num_features
        self.plane_emb = nn.Parameter(torch.zeros(N_PLANES, f))
        self.norm = nn.LayerNorm(f)
        self.att = nn.Sequential(nn.Linear(f, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n_out))
        self.cls_w = nn.Parameter(torch.randn(n_out, f) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(n_out))
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x, present):
        """x: (B, P, K, 3, H, W) in [0, 1]; present: (B, P) -> logits (B, n_out)."""
        b, p, k = x.shape[:3]
        feats = self.backbone((x.flatten(0, 2) - self.mean) / self.std)
        h = feats.view(b, p, k, -1).float() + self.plane_emb[None, :, None]
        h = self.norm(h.flatten(1, 2))                                  # (B, P*K, F)
        a = self.att(h).float()                                         # (B, P*K, n)
        keep = present[:, :, None].expand(b, p, k).reshape(b, p * k)
        a = torch.softmax(a.masked_fill(keep[..., None] <= 0, -1e4), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        return (pooled * self.cls_w).sum(-1) + self.cls_b

In [ ]:
MODEL_KIND = config.get("model_kind", "linear")  # 14's bundle predates the key
assert PREPROC_VERSION == config["preproc_version"], (PREPROC_VERSION, config["preproc_version"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if MODEL_KIND == "mil":
    assert MIL_VERSION == config["mil_version"], (MIL_VERSION, config["mil_version"])
    assert not config["smoke"], "this bundle is from a SMOKE run of notebook 16"
    net = KneeMIL(config["arch"], pretrained=False)  # weights come from the bundle; internet is off
    net.load_state_dict(torch.load(MODEL / "mil_weights.pt", map_location="cpu"))  # fp16 -> fp32 on copy
    net = net.to(device).eval()
    centres = [eval_centres(config["k_eval"])] * N_PLANES

    @torch.no_grad()
    def predict_study(vol, present):
        x = torch.from_numpy(windows_from_volume(vol, centres))[None].to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            logits = net(x, torch.from_numpy(present.astype(np.float32))[None].to(device))
        return torch.sigmoid(logits.float())[0].cpu().numpy()
    print("model: MIL", config["arch"], "| k_eval:", config["k_eval"], "| device:", device)
elif MODEL_KIND == "linear":
    assert FEATURE_VERSION == config["feature_version"], (FEATURE_VERSION, config["feature_version"])
    encoder = build_encoder(torch.load(MODEL / "encoder_resnet18.pt", map_location="cpu")).to(device)
    heads = []
    for info in config["folds"]:
        b = torch.load(MODEL / f"head_fold{info['fold']}.pt", map_location="cpu", weights_only=False)
        h = nn.Linear(3 * 512 + 3, 12)
        h.load_state_dict(b["state_dict"])
        heads.append((h.eval(), b["mean"], b["std"]))

    def predict_study(vol, present):
        x = study_features(encoder, torch.from_numpy(vol)[None], torch.from_numpy(present)[None],
                           device).numpy()
        with torch.no_grad():
            return np.mean([torch.sigmoid(h(torch.from_numpy(((x - m) / s).astype(np.float32)))).numpy()[0]
                            for h, m, s in heads], axis=0)
    print("model: linear | fold heads:", len(heads), "| device:", device)
else:
    raise ValueError(f"unknown model_kind {MODEL_KIND!r}")

series = pd.read_csv(ROOT / "test_series.csv", dtype=str)
for c in ("Fluid_Sensitive", "Fat_Suppression"):
    series[c] = series[c].astype(int)
by_study = dict(tuple(series.groupby("StudyInstanceUID")))

In [ ]:
rows, log = [], []
from tqdm.auto import tqdm
for study in tqdm(test["StudyInstanceUID"], desc="test studies", mininterval=30):
    try:
        vol, present, rec = process_study(by_study[study], ROOT / "test_series")
        if not present.any():
            raise ValueError("no usable series")
        rows.append([study, *predict_study(vol, present).tolist()])
        log.append({"status": "ok", **rec})
    except Exception as exc:
        rows.append([study, *[config["fallback_prevalence"][t] for t in TARGET_COLUMNS]])
        log.append({"status": "fallback", "error": f"{type(exc).__name__}: {exc}"})

sub = pd.DataFrame(rows, columns=["StudyInstanceUID"] + TARGET_COLUMNS)
log = pd.DataFrame(log)
print("status:", log["status"].value_counts().to_dict())

## Schema checks, then write

In [ ]:
assert list(sub.columns) == list(sample.columns)
assert sub["StudyInstanceUID"].tolist() == test["StudyInstanceUID"].tolist()
values = sub[TARGET_COLUMNS].to_numpy()
assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()
sub.to_csv(OUT / "submission.csv", index=False)
print("submission.csv:", sub.shape)